# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/techwolfdarim/FlyRank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type:** **Classification** (specifically binary classification: identifying content items that are declining / need refresh attention versus those that are not).

**Why classification fits:** My lane focuses on deciding whether a content item falls into a declining/refresh-needed state so a content editor or strategist can decide which pages to **queue for a content refresh or audit** and which ones can be left alone. A wrong call has a clear tradeoff: flagging healthy pages wastes limited editorial hours, while missing declining pages delays recovery and costs search/engagement value.

In [4]:
from pathlib import Path
import subprocess
import pandas as pd

REPO_URL = "https://github.com/techwolfdarim/FlyRank-Internship.git"  # update if your repo URL differs
REPO_DIR = Path("/content/FlyRank-Internship")
TARGET_NAME = "content_refresh_anonymized.csv"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

matches = list(REPO_DIR.rglob(TARGET_NAME))
if not matches:
    raise FileNotFoundError(f"Cloned {REPO_DIR}, but could not find {TARGET_NAME}.")

DATA_PATH = matches[0]
print("Loaded from:", DATA_PATH.resolve())
df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
if "is_declining_label" in df.columns:
    print("Base rate (is_declining_label mean):", round(df["is_declining_label"].mean(), 4))
    print(df["is_declining_label"].value_counts(dropna=False))

Loaded from: /content/FlyRank-Internship/data/raw/content_refresh_anonymized.csv
Rows: 30,000 | Columns: 44


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target / Proxy:** Predict whether a content item is declining (`is_declining_label` as a starting/prototype target in the starter CSV, or an observed future decline window in later warehouse work).

**Where it comes from (observed outcome vs. defined rule):** In the starter dataset, `is_declining_label` is derived from `trend_direction` (which itself comes from `trend_pct`). Because a label derived from a rule/trend calculation reflects that definition rather than a separate downstream outcome, I must treat `trend_direction` and `trend_pct` as **strictly forbidden features** (never feed them into the model, or the model just learns the formula). In an ideal setup, a target measured over a later time window—or paired with honest proxy caveats—gives cleaner signal than a formula-derived label.

In [5]:
forbidden_cols = [c for c in ["is_declining_label", "trend_direction", "trend_pct"] if c in df.columns]
print("Label & forbidden feature columns present:", forbidden_cols)

for col in ["trend_direction", "is_declining_label"]:
    if col in df.columns:
        print(f"\nValue counts for {col}:")
        print(df[col].value_counts(dropna=False))

Label & forbidden feature columns present: ['trend_direction', 'trend_pct']

Value counts for trend_direction:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success metric:** **Precision and Recall compared against the base rate** (with **ROC-AUC** as a supporting rank-order check, and **Precision@K** if we treat the top-scored items as a fixed editorial refresh queue).

**Why this metric & what "good" looks like:** Accuracy alone can look high on imbalanced data just by predicting the majority class. Comparing precision and recall against the **base rate** (the share of positive/declining rows in the dataset) shows whether the model actually beats chance. A "good" result means **meaningfully higher precision/recall than the base rate** (and solid ROC-AUC) on a **grouped split by `client_id`**—with exact numerical thresholds set as directional / decision-support benchmarks once baseline validation runs are logged.

In [6]:
if "is_declining_label" in df.columns:
    y = df["is_declining_label"].dropna()
    base_rate = y.mean()
    print(f"Observed positive base rate: {base_rate:.4f} ({base_rate * 100:.2f}%)")
    print(f"Dummy 'always-0' accuracy would be: {(1 - base_rate):.4f} ({(1 - base_rate) * 100:.2f}%)")
    print(f"Dummy 'always-1' precision baseline equals base rate: {base_rate:.4f}")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis:** **One row = one pseudonymized content item** (across 32 clients in the trailing-90-day starter dataset).

**Data hygiene & target sketch:** Keep `content_id` and `client_id` for grouping/joining/splitting only—never as features—and split by `client_id` to prevent client leakage. Also account for dataset gotchas: `avg_position == 0` means no data (not rank 0), percentage columns (`ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `trend_pct`) are already scaled by 100 (and can exceed 100), and missingness depends on `content_type`, so I add `has_*` missingness flags rather than blindly filling zeros. Below is a DataFrame showing the unit of analysis and a sketched target column.

In [7]:
unit_cols = [c for c in ["content_id", "client_id", "content_type"] if c in df.columns]
candidate_metrics = [
    c for c in [
        "clicks", "impressions", "ctr", "avg_position",
        "engagement_rate", "scroll_rate", "ai_traffic_pct", "word_count"
    ]
    if c in df.columns
]

lane_df = df[unit_cols + candidate_metrics].copy()

if "avg_position" in lane_df.columns:
    lane_df["has_position_data"] = (lane_df["avg_position"] > 0).astype(int)
if "word_count" in lane_df.columns:
    lane_df["has_word_count"] = lane_df["word_count"].notna().astype(int)

# Sketched target column for inspection (do not use trend_pct / trend_direction as features)
if "is_declining_label" in df.columns:
    lane_df["target_declining_sketch"] = df["is_declining_label"]
else:
    lane_df["target_declining_sketch"] = pd.NA

print("Unique content_id count:", df["content_id"].nunique() if "content_id" in df.columns else "N/A")
print("Unique client_id count:", df["client_id"].nunique() if "client_id" in df.columns else "N/A")
if "avg_position" in df.columns:
    print("Rows with avg_position == 0 (no data):", int((df["avg_position"] == 0).sum()))

lane_df.head(10)

Unique content_id count: 30000
Unique client_id count: 32
Rows with avg_position == 0 (no data): 1205


,content_id,client_id,content_type,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,word_count,has_position_data,has_word_count,target_declining_sketch
0,content_304f48230142,client_f369cb89fc,keyword article,0.76,10.6,5.88,4.55,0.0,3221.0,1,1,<NA>
1,content_a1fb4e703a9e,client_4e07408562,keyword article,0.05,20.3,0.00,10.00,0.0,2481.0,1,1,<NA>
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,0.09,36.5,0.00,28.57,0.0,3515.0,1,1,<NA>
3,content_331d6c4de07b,client_19581e27de,keyword article,0.49,6.2,1.28,3.45,0.0,NaN,1,0,<NA>
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,0.13,44.0,0.00,24.29,0.0,2803.0,1,1,<NA>
5,content_d4084a4bc775,client_f369cb89fc,keyword article,0.03,8.5,0.00,25.00,0.0,3080.0,1,1,<NA>
6,content_9a34b442b552,client_8722616204,keyword article,0.00,7.0,0.00,0.00,0.0,3059.0,1,1,<NA>
7,content_a63219c6e95a,client_19581e27de,keyword article,0.06,21.2,3.57,7.14,0.0,NaN,1,0,<NA>
8,content_5e6c160719bc,client_6208ef0f77,keyword article,0.09,46.0,5.88,6.25,0.0,3807.0,1,1,<NA>
9,content_c27558df2b0c,client_19581e27de,keyword article,0.16,4.9,0.00,0.00,0.0,NaN,1,0,<NA>


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why ML beats a fixed rule:** A simple `if/else` rule (such as "flag for refresh if CTR is below X or clicks are below Y") breaks down because content performance depends on **multiple interacting signals** (`impressions`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`) that behave differently across `content_type` and `client_id`. Hardcoded thresholds either miss subtle multi-signal declines or trigger false alarms on normal differences between content formats and missing-data patterns. An ML model learns combined decision boundaries from data (and serves as a step up from a simple rule baseline to test whether learned patterns add real value).

In [8]:
summary_cols = [c for c in ["clicks", "impressions", "ctr", "avg_position", "engagement_rate"] if c in df.columns]
if "is_declining_label" in df.columns and summary_cols:
    grouped_means = df.groupby("is_declining_label")[summary_cols].mean(numeric_only=True)
    display(grouped_means.round(3))

if "content_type" in df.columns:
    print("\nMissingness rate by content_type (top columns with missing values):")
    missing_by_type = df.groupby("content_type").apply(
        lambda g: g.isna().mean(), include_groups=False
    )
    cols_with_na = [c for c in df.columns if df[c].isna().any()]
    if cols_with_na:
        display(missing_by_type[cols_with_na].round(3))


Missingness rate by content_type (top columns with missing values):


,search_volume,competition,competition_level,cpc,main_intent,word_count,char_count,provider_used,model_used,word_count_tier,char_count_tier,scroll_rate,trend_pct
content_type,,,,,,,,,,,,,
comparison article,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.834,0.000,0.000,0.000,0.003,0.003
feedly article,1.000,1.000,1.000,1.000,1.00,0.000,0.000,0.700,0.000,0.000,0.000,0.000,0.519
keyword article,0.014,0.014,0.019,0.014,0.01,0.283,0.283,0.713,0.211,0.283,0.283,0.004,0.085


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.